# Retinal Vessel Segmentation using FR-UNet Ensemble


**This notebook trains an ensemble of FR-UNet models on the FIVES dataset for robust retinal vessel segmentation.We train multiple models with different random seeds to later enableuncertainty-based quality estimation and patch-level correction.**


In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [2]:
FIVES_BASE = "/kaggle/input/datasets/nikitamanaenkov/fundus-image-dataset-for-vessel-segmentation"

FIVES_TRAIN_IMG = os.path.join(FIVES_BASE, "train/Original")
FIVES_TRAIN_MASK = os.path.join(FIVES_BASE, "train/Ground truth")

FIVES_TEST_IMG = os.path.join(FIVES_BASE, "test/Original")
FIVES_TEST_MASK = os.path.join(FIVES_BASE, "test/Ground truth")

In [3]:
!git clone https://github.com/berenslab/MIDL24-segmentation_quality_control.git

fatal: destination path 'MIDL24-segmentation_quality_control' already exists and is not an empty directory.


In [4]:
%cd MIDL24-segmentation_quality_control

/kaggle/working/MIDL24-segmentation_quality_control


In [5]:
!pip install albumentations opencv-python tqdm scikit-image

In [6]:
class VesselDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        
        self.images = [
            f for f in os.listdir(img_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif'))
        ]
        self.images.sort()

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        img = cv2.imread(img_path)

        if img is None:
            raise ValueError(f"Image not found or corrupted: {img_path}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (512, 512))
        img = img / 255.0
        img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)

        mask = cv2.imread(mask_path, 0)

        if mask is None:
            raise ValueError(f"Mask not found: {mask_path}")

        mask = cv2.resize(mask, (512, 512))
        mask = mask / 255.0
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        return img, mask

In [7]:
train_dataset = VesselDataset(FIVES_TRAIN_IMG, FIVES_TRAIN_MASK)
test_dataset = VesselDataset(FIVES_TEST_IMG, FIVES_TEST_MASK)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [8]:
import sys
sys.path.append("/kaggle/working/MIDL24-segmentation_quality_control")

from models.frunet import FR_UNet

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [9]:
%%writefile /kaggle/working/MIDL24-segmentation_quality_control/models/frunet.py
import torch
import torch.nn as nn
from timm.models.layers import trunc_normal_

# =====================================================
# Weight Initialization
# =====================================================
class InitWeights_He(object):

    def __init__(self, neg_slope=1e-2):
        self.neg_slope = neg_slope

    def __call__(self, module):

        if isinstance(module,
                      (nn.Conv2d,
                       nn.ConvTranspose2d)):

            nn.init.kaiming_normal_(
                module.weight,
                a=self.neg_slope
            )

            if module.bias is not None:
                nn.init.constant_(module.bias, 0)

        elif isinstance(module, nn.Linear):

            trunc_normal_(module.weight,
                          std=self.neg_slope)

            if module.bias is not None:
                nn.init.constant_(module.bias, 0)

        elif isinstance(module, nn.BatchNorm2d):

            nn.init.constant_(module.weight, 1.0)
            nn.init.constant_(module.bias, 0)


# =====================================================
# Residual Conv Block + MC Dropout
# =====================================================
class ConvBlock(nn.Module):

    def __init__(self, in_c, out_c, dropout=0.3):
        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout2d(dropout),

            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout2d(dropout),
        )

        self.residual = nn.Conv2d(in_c, out_c, 1)

    def forward(self, x):

        res = self.residual(x)
        x = self.conv(x)

        return x + res


# =====================================================
# Down Block
# =====================================================
class Down(nn.Module):

    def __init__(self, in_c, out_c, dropout):
        super().__init__()

        self.conv = ConvBlock(in_c, out_c, dropout)

        self.pool = nn.Sequential(
            nn.Conv2d(out_c, out_c,
                      kernel_size=2,
                      stride=2),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.1)
        )

    def forward(self, x):

        x = self.conv(x)
        down = self.pool(x)

        return x, down


# =====================================================
# Up Block
# =====================================================
class Up(nn.Module):

    def __init__(self, in_c, out_c, dropout):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_c,
            out_c,
            kernel_size=2,
            stride=2
        )

        self.conv = ConvBlock(
            in_c,
            out_c,
            dropout
        )

    def forward(self, x, skip):

        x = self.up(x)
        x = torch.cat([x, skip], dim=1)

        return self.conv(x)


# =====================================================
# FR-UNet with MC Dropout
# =====================================================
class FR_UNet(nn.Module):

    def __init__(
        self,
        num_classes=1,
        num_channels=3,
        dropout=0.3
    ):

        super().__init__()

        filters = [32, 64, 128, 256]

        # Encoder
        self.d1 = Down(num_channels,
                       filters[0],
                       dropout)

        self.d2 = Down(filters[0],
                       filters[1],
                       dropout)

        self.d3 = Down(filters[1],
                       filters[2],
                       dropout)

        # Bottleneck
        self.bridge = ConvBlock(
            filters[2],
            filters[3],
            dropout
        )

        # Decoder
        self.u3 = Up(filters[3],
                     filters[2],
                     dropout)

        self.u2 = Up(filters[2],
                     filters[1],
                     dropout)

        self.u1 = Up(filters[1],
                     filters[0],
                     dropout)

        # Output
        self.final = nn.Conv2d(
            filters[0],
            num_classes,
            kernel_size=1
        )

        self.apply(InitWeights_He())

    # -------------------------------------------------
    # IMPORTANT → Enable Dropout During Inference
    # -------------------------------------------------
    def enable_dropout(self):

        for m in self.modules():
            if isinstance(m, nn.Dropout2d):
                m.train()

    # -------------------------------------------------
    def forward(self, x):

        s1, x = self.d1(x)
        s2, x = self.d2(x)
        s3, x = self.d3(x)

        x = self.bridge(x)

        x = self.u3(x, s3)
        x = self.u2(x, s2)
        x = self.u1(x, s1)

        return self.final(x)
    def enable_dropout(self):

        for m in self.modules():
            if isinstance(m, nn.Dropout2d):
                m.train()

Overwriting /kaggle/working/MIDL24-segmentation_quality_control/models/frunet.py


In [10]:
from models.frunet import FR_UNet
import torch

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FR_UNet(
    num_classes=1,
    num_channels=3,
    dropout=0.3   # ✅ MC Dropout ACTIVE
).to(device)

In [12]:
model.eval()
model.enable_dropout()

In [13]:
x = torch.randn(1,3,512,512).to(device)

out1 = model(x)
out2 = model(x)

print(torch.mean(torch.abs(out1 - out2)))

tensor(71.0175, device='cuda:0', grad_fn=<MeanBackward0>)


In [14]:
import torch
import random
import numpy as np

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [15]:
import torch
import torch.nn as nn
from tqdm import tqdm

def dice_loss(pred, target, smooth=1e-6):

    pred = torch.sigmoid(pred)

    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()

    dice = (2. * intersection + smooth) / \
           (pred.sum() + target.sum() + smooth)

    return 1 - dice

In [16]:
def dice_score(pred, target, smooth=1e-6):

    pred = (pred > 0.5).float()

    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()

    dice = (2. * intersection + smooth) / \
           (pred.sum() + target.sum() + smooth)

    return dice

In [17]:
def train_model(model,
                loader,
                epochs=30,
                lr=1e-4):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    model.train()

    for epoch in range(epochs):

        total_loss = 0
        total_dice = 0

        loop = tqdm(loader)

        for imgs, masks in loop:

            imgs = imgs.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()

            outputs = model(imgs)

            loss = dice_loss(outputs, masks)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            dice = dice_score(
                torch.sigmoid(outputs),
                masks
            )

            total_dice += dice.item()

            loop.set_description(
                f"Epoch [{epoch+1}/{epochs}]"
            )

        print(
            f"Loss: {total_loss/len(loader):.4f} | "
            f"Dice: {total_dice/len(loader):.4f}"
        )

In [18]:
models = []

for i in range(5):

    print(f"\nTraining Model {i}")

    set_seed(i * 100)

    model = FR_UNet(
        num_classes=1,
        num_channels=3,
        dropout=0.3     # ⭐ MC Dropout ACTIVE
    ).to(device)

    train_model(
        model,
        train_loader,
        epochs=30
    )

    torch.save(
        model.state_dict(),
        f"FRUNet_MC_{i}.pth"
    )

    models.append(model)


Training Model 0


Epoch [1/30]: 100%|██████████| 150/150 [02:54<00:00,  1.16s/it]


Loss: 0.7940 | Dice: 0.2087


Epoch [2/30]: 100%|██████████| 150/150 [02:41<00:00,  1.08s/it]


Loss: 0.5184 | Dice: 0.4861


...

Loss: 0.1687 | Dice: 0.8321


Epoch [29/30]: 100%|██████████| 150/150 [02:41<00:00,  1.08s/it]


Loss: 0.1646 | Dice: 0.8361


Epoch [30/30]: 100%|██████████| 150/150 [02:42<00:00,  1.08s/it]

Loss: 0.1622 | Dice: 0.8385
